# Soluzione 4 - Output piu adatto a un LLM

**Obiettivo:** convertire lo stesso `AnalysisResult` con il metodo ufficiale
`to_llm_input`, senza appiattire a mano campi, pagine e tabelle.
Prepara i blocchi **1 e 3 di 1**, **3 di 2**, **1 di 3**.


## Preparazione - Percorsi del laboratorio

Blocco ripetuto per poter partire da un kernel nuovo.

In [ ]:
from pathlib import Path
import hashlib
import json
import os

ROOT = Path.cwd().resolve()
if ROOT.name in {"soluzioni", "esercizi"}:
    ROOT = ROOT.parent
if not (ROOT / ".env.example").is_file() or not (ROOT / "esercizi").is_dir():
    raise RuntimeError("Avvia Jupyter dalla root della repo o da soluzioni.")

OUTPUT = ROOT / "soluzioni" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
ENV_PATH = ROOT / ".env"
print("Root:", ROOT)

## Preparazione - Caricamento di .env e controllo dei segreti

Blocco ripetuto per poter partire da un kernel nuovo.

In [ ]:
from dotenv import load_dotenv

if not ENV_PATH.is_file():
    raise FileNotFoundError("Manca .env: completa l'esercizio 1.")
load_dotenv(ENV_PATH, override=True)

def required_env(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value or "<" in value or ">" in value:
        raise ValueError(f"Compila {name} in .env, poi riesegui questa cella.")
    return value

CU_ENDPOINT = required_env("AZURE_CONTENTUNDERSTANDING_ENDPOINT")
CU_KEY = required_env("AZURE_CONTENTUNDERSTANDING_API_KEY")
if not CU_ENDPOINT.startswith("https://") or "/api/projects/" in CU_ENDPOINT:
    raise ValueError("CU richiede l'endpoint HTTPS della risorsa, non del progetto.")
print("Configurazione CU caricata; chiave non visualizzata.")

## Preparazione - Analyzer salvato nell'esercizio 2

Blocco ripetuto per poter partire da un kernel nuovo.

In [ ]:
selection_path = OUTPUT / "analyzer-selection.json"
if not selection_path.is_file():
    raise FileNotFoundError("Completa prima l'esercizio 2: manca la selezione analyzer.")
selection = json.loads(selection_path.read_text(encoding="utf-8"))
if selection["endpoint"] != CU_ENDPOINT.rstrip("/"):
    raise ValueError("L'analyzer salvato appartiene a un altro endpoint. Riesegui 2.")
ANALYZER_ID = selection["analyzer_id"]
print("Analyzer selezionato:", ANALYZER_ID)

## Preparazione - Lettura del documento locale e impronta SHA-256

Blocco ripetuto per poter partire da un kernel nuovo.

In [ ]:
DOCUMENT_PATH = Path(required_env("DOCUMENT_PATH")).expanduser()
if not DOCUMENT_PATH.is_absolute():
    DOCUMENT_PATH = ROOT / DOCUMENT_PATH
DOCUMENT_PATH = DOCUMENT_PATH.resolve()
if not DOCUMENT_PATH.is_file():
    raise FileNotFoundError(f"Documento non trovato: {DOCUMENT_PATH}")

MEDIA_TYPES = {
    ".pdf": "application/pdf", ".png": "image/png",
    ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
}
if DOCUMENT_PATH.suffix.lower() not in MEDIA_TYPES:
    raise ValueError("Per questo laboratorio usa PDF, PNG o JPEG.")
MEDIA_TYPE = MEDIA_TYPES[DOCUMENT_PATH.suffix.lower()]
document_bytes = DOCUMENT_PATH.read_bytes()
if not document_bytes:
    raise ValueError("Il documento e vuoto.")
DOCUMENT_SHA256 = hashlib.sha256(document_bytes).hexdigest()
print("Documento:", DOCUMENT_PATH.name, "-", len(document_bytes), "byte")

## 1. Riusa l'analisi dello stesso documento

Ricarichiamo 3 dal disco: l'analisi e gia stata eseguita, questa trasformazione
non richiede un'altra chiamata Azure. Verifichiamo hash, analyzer ed endpoint.

In [ ]:
from azure.ai.contentunderstanding.models import AnalysisResult

result_path = OUTPUT / "analisi.json"
input_path = OUTPUT / "analisi-input.json"
if not result_path.is_file() or not input_path.is_file():
    raise FileNotFoundError("Esegui prima l'esercizio 3.")
analyzed_input = json.loads(input_path.read_text(encoding="utf-8"))
expected_input = {
    "sha256": DOCUMENT_SHA256,
    "analyzer_id": ANALYZER_ID,
    "endpoint": CU_ENDPOINT.rstrip("/"),
}
if any(analyzed_input.get(key) != value for key, value in expected_input.items()):
    raise ValueError("Documento, analyzer o endpoint cambiato: riesegui 3.")
result = AnalysisResult(json.loads(result_path.read_text(encoding="utf-8")))
if not result.contents:
    raise ValueError("Il risultato salvato non contiene contenuti.")

## 2. Facoltativo: ripeti davvero la chiamata di analisi

Lascia `False` per riusare il risultato di 3. Impostalo a `True` solo se vuoi
rianalizzare lo stesso file **con ulteriori costi CU** e confrontare eventuali
variazioni. Il risultato nuovo resta in memoria e non sovrascrive l'originale di 3.

In [ ]:
RIPETI_ANALISI = False
if RIPETI_ANALISI:
    from azure.ai.contentunderstanding import ContentUnderstandingClient
    from azure.core.credentials import AzureKeyCredential
    with ContentUnderstandingClient(
        endpoint=CU_ENDPOINT, credential=AzureKeyCredential(CU_KEY)
    ) as client:
        result = client.begin_analyze_binary(
            analyzer_id=ANALYZER_ID, binary_input=document_bytes,
        ).result()
    if not result.contents:
        raise RuntimeError("La nuova analisi non contiene contenuti.")

## 3. Converti con to_llm_input

Il risultato e una **stringa** con front matter YAML (campi e metadati) e corpo
Markdown (testo, tabelle, riferimenti alle pagine). Non e una risposta di un LLM,
non esegue inferenza e non garantisce automaticamente un limite di token.
Il JSON dettagliato di 3 resta la fonte per confidence, coordinate e debugging.

In [ ]:
from azure.ai.contentunderstanding import to_llm_input

llm_text = to_llm_input(result)
fields_only = to_llm_input(result, include_markdown=False)
markdown_only = to_llm_input(result, include_fields=False)
if not llm_text.strip():
    raise RuntimeError("Conversione LLM vuota.")

(OUTPUT / "contesto-llm.md").write_text(llm_text, encoding="utf-8")
print(llm_text)
print("\nCaratteri (NON token):", {
    "json": len(json.dumps(result.as_dict(), ensure_ascii=False)),
    "llm_completo": len(llm_text),
    "solo_campi": len(fields_only),
    "solo_markdown": len(markdown_only),
})

## Cosa verificare

- Il testo mantiene `Ospite`, totali e saldo senza confonderli.
- `fields_only` esclude il corpo Markdown; `markdown_only` esclude i campi.
- La conversione e diversa da `json.dumps(result.as_dict())` o da
  `result.contents[0].markdown`: questi due approcci non fanno lo stesso lavoro.
- Nell'esercizio 6 sara il provider a fare questa preparazione automaticamente;
  non incolleremo a mano `contesto-llm.md` nel prompt.